# Сравнение методов рекомендаций поставщиков

1. **Cold-start** — мало истории в train (< 10 транзакций), но активны в test.
2. **Hot+** — расширенное определение: появился новый поставщик (доля ≥ 10% в test, < 3% в train) **ИЛИ** сменился топ-1 поставщик **ИЛИ** Δ wallet share ≥ 25%.
3. **Inertial filter** — клиенты, у которых состав и порядок поставщиков почти не изменился, **исключаются** из обучения и из основной оценки.

In [ ]:
import sys
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from collaborative_experiments import (
    load_clients, load_transactions, clean_transactions, temporal_split,
    build_all_features, build_target,
    MostPopRecommender, ALSRecommender, EASERecommender
)
from extra_models import CatBoostLikeRanker
from models_and_validation import ContentBasedRecommender

SPLIT_DATE = '2025-05-15'

ALPHA_FREQ = 0.5
ALPHA_VOL = 0.5
MIN_RELEVANCE_SHARE = 0.10
MIN_TX_FOR_EVAL = 3

#Параметры фильтрации потенциальных клиентов 
COLD_TX_THRESHOLD = 10        # < N транзакций в train → cold-start
HOT_DELTA_SHARE = 0.25        # Δ wallet share >= 25% → hot
NEW_SUPPLIER_TEST = 0.10      # Доля нового поставщика в test
NEW_SUPPLIER_TRAIN = 0.03     # Доля того же поставщика в train

ROLLING_WINDOW_DAYS = 30

In [5]:
print("Загрузка данных...")
clients = load_clients()
tx = load_transactions()

if "Поставщик" not in tx.columns:
    tx["Поставщик"] = tx["Тип карты"].str.extract(r'(Поставщик\d+)')

tx_train, tx_test = temporal_split(tx, SPLIT_DATE)
tx_test_clean = clean_transactions(tx_test)
print(f"Train: {len(tx_train)} транзакций, Test: {len(tx_test_clean)} транзакций")

print("\nПостроение признаков на train...")
tf = build_all_features(tx_train, clients)
tt = build_target(
    tf["compatibility_features"],
    tf["supplier_profile"],
    tf["interaction_features"],
    tf["client_profile"]
)
print("Готово")

Загрузка данных...
Train: 574495 транзакций, Test: 214409 транзакций

Построение признаков на train...
Готово


## 1. Ground Truth

In [6]:
def build_ground_truth_from_test(tx_test, alpha_vol=0.5, alpha_freq=0.5,
                                  min_share=0.10, min_tx=3):
    tx_per_client = tx_test.groupby("Код клиента").size()
    eligible = tx_per_client[tx_per_client >= min_tx].index
    tx_filtered = tx_test[tx_test["Код клиента"].isin(eligible)].copy()
    
    if "Поставщик" not in tx_filtered.columns:
        tx_filtered["Поставщик"] = tx_filtered["Тип карты"].str.extract(r'(Поставщик\d+)')
    
    agg = tx_filtered.groupby(["Код клиента", "Поставщик"]).agg(
        volume=("Объем", "sum"), txn=("Объем", "count")
    ).reset_index()
    
    tot_vol = agg.groupby("Код клиента")["volume"].transform("sum")
    tot_txn = agg.groupby("Код клиента")["txn"].transform("sum")
    agg["vol_share"] = agg["volume"] / tot_vol
    agg["txn_share"] = agg["txn"] / tot_txn
    agg["relevance"] = alpha_vol * agg["vol_share"] + alpha_freq * agg["txn_share"]
    agg = agg[agg["relevance"] >= min_share].copy()
    agg = agg.sort_values(["Код клиента", "relevance"], ascending=[True, False])
    
    gt_list = agg.groupby("Код клиента").apply(
        lambda x: pd.Series({
            "true_suppliers": x["Поставщик"].tolist(),
            "relevance_dict": dict(zip(x["Поставщик"], x["relevance"])),
            "n_suppliers": len(x)
        })
    ).reset_index()
    return gt_list

gt = build_ground_truth_from_test(tx_test_clean, ALPHA_VOL, ALPHA_FREQ,
                                   MIN_RELEVANCE_SHARE, MIN_TX_FOR_EVAL)
train_clients_set = set(tf["client_profile"]["Код клиента"])
gt = gt[gt["Код клиента"].isin(train_clients_set)].reset_index(drop=True)
print(f"Клиентов в GT: {len(gt)}")

Клиентов в GT: 3536


## 2. Сегментация: потенциальные vs инертные

In [7]:
def compute_wallet_share_matrix(tx_df):
    if "Поставщик" not in tx_df.columns:
        tx_df = tx_df.copy()
        tx_df["Поставщик"] = tx_df["Тип карты"].str.extract(r'(Поставщик\d+)')
    vol = tx_df.groupby(["Код клиента", "Поставщик"])["Объем"].sum().unstack(fill_value=0)
    return vol.div(vol.sum(axis=1), axis=0).fillna(0)

share_train = compute_wallet_share_matrix(tx_train)
share_test = compute_wallet_share_matrix(tx_test_clean)

all_suppliers = sorted(set(share_train.columns) | set(share_test.columns))
share_train = share_train.reindex(columns=all_suppliers, fill_value=0)
share_test = share_test.reindex(columns=all_suppliers, fill_value=0)

common_clients = share_train.index.intersection(share_test.index)
share_train_c = share_train.loc[common_clients]
share_test_c = share_test.loc[common_clients]
delta_share = share_test_c - share_train_c

potential_flags = pd.DataFrame(index=common_clients)
potential_flags["new_supplier"] = ((share_train_c < NEW_SUPPLIER_TRAIN) & (share_test_c >= NEW_SUPPLIER_TEST)).any(axis=1)
potential_flags["lost_supplier"] = ((share_train_c >= NEW_SUPPLIER_TEST) & (share_test_c < NEW_SUPPLIER_TRAIN)).any(axis=1)

top1_train = share_train_c.idxmax(axis=1)
top1_test = share_test_c.idxmax(axis=1)
potential_flags["top1_change"] = (top1_train != top1_test) & (share_test_c.max(axis=1) > 0)
potential_flags["big_shift"] = delta_share.abs().max(axis=1) >= HOT_DELTA_SHARE
potential_flags["is_potential"] = potential_flags.any(axis=1)

tx_per_client_train = tx_train.groupby("Код клиента").size()
cold_clients_set = set(tx_per_client_train[tx_per_client_train < COLD_TX_THRESHOLD].index)

print("=== Распределение признаков потенциальности ===")
print(f"Всего клиентов в train ∩ test: {len(common_clients)}")
print(f"  Появился новый поставщик:  {potential_flags['new_supplier'].sum()} ({100*potential_flags['new_supplier'].mean():.1f}%)")
print(f"  Ушёл поставщик:            {potential_flags['lost_supplier'].sum()} ({100*potential_flags['lost_supplier'].mean():.1f}%)")
print(f"  Сменился топ-1:            {potential_flags['top1_change'].sum()} ({100*potential_flags['top1_change'].mean():.1f}%)")
print(f"  Δ wallet share ≥ {HOT_DELTA_SHARE*100:.0f}%:    {potential_flags['big_shift'].sum()} ({100*potential_flags['big_shift'].mean():.1f}%)")
print(f"  Cold-start (< {COLD_TX_THRESHOLD} tx):       {len(cold_clients_set)}")
print(f"\n  ИТОГО потенциальных:        {potential_flags['is_potential'].sum()} ({100*potential_flags['is_potential'].mean():.1f}%)")

=== Распределение признаков потенциальности ===
Всего клиентов в train ∩ test: 3697
  Появился новый поставщик:  42 (1.1%)
  Ушёл поставщик:            90 (2.4%)
  Сменился топ-1:            134 (3.6%)
  Δ wallet share ≥ 25%:    125 (3.4%)
  Cold-start (< 10 tx):       446

  ИТОГО потенциальных:        255 (6.9%)


In [8]:
potential_warm_set = set(potential_flags[potential_flags["is_potential"]].index)
all_potential_set = potential_warm_set | cold_clients_set
inertial_set = set(common_clients) - all_potential_set

gt_potential = gt[gt["Код клиента"].isin(all_potential_set)].reset_index(drop=True)
gt_inertial = gt[gt["Код клиента"].isin(inertial_set)].reset_index(drop=True)
gt_cold = gt[gt["Код клиента"].isin(cold_clients_set)].reset_index(drop=True)
gt_hot_plus = gt[gt["Код клиента"].isin(potential_warm_set)].reset_index(drop=True)

print("=== Сегменты для оценки V3 ===")
print(f"  Все клиенты в GT:           {len(gt)}")
print(f"  Инертные:                   {len(gt_inertial)}")
print(f"  ПОТЕНЦИАЛЬНЫЕ (cold + hot+): {len(gt_potential)}  ← главная выборка")
print(f"    из них Cold-start:        {len(gt_cold)}")
print(f"    из них Hot+:              {len(gt_hot_plus)}")

=== Сегменты для оценки V3 ===
  Все клиенты в GT:           3536
  Инертные:                   3134
  ПОТЕНЦИАЛЬНЫЕ (cold + hot+): 402  ← главная выборка
    из них Cold-start:        166
    из них Hot+:              244


## 3. Метрики ранжирования

In [9]:
def _prepare_topk(predictions, k):
    df = predictions.sort_values(["Код клиента", "score"], ascending=[True, False]).copy()
    df["rank"] = df.groupby("Код клиента").cumcount() + 1
    return df[df["rank"] <= k]

def hit_rate_at_k(predictions, ground_truth, k=1):
    topk = _prepare_topk(predictions, k)
    gt_dict = ground_truth.set_index("Код клиента")["true_suppliers"].to_dict()
    hits = []
    for cid, group in topk.groupby("Код клиента"):
        if cid not in gt_dict: continue
        hits.append(len(set(group["Поставщик"].values) & set(gt_dict[cid])) > 0)
    return np.mean(hits) if hits else 0.0

def precision_at_k(predictions, ground_truth, k=3):
    topk = _prepare_topk(predictions, k)
    gt_dict = ground_truth.set_index("Код клиента")["true_suppliers"].to_dict()
    precs = []
    for cid, group in topk.groupby("Код клиента"):
        if cid not in gt_dict: continue
        precs.append(len(set(group["Поставщик"].values) & set(gt_dict[cid])) / k)
    return np.mean(precs) if precs else 0.0

def recall_at_k(predictions, ground_truth, k=3):
    topk = _prepare_topk(predictions, k)
    gt_dict = ground_truth.set_index("Код клиента")["true_suppliers"].to_dict()
    recs = []
    for cid, group in topk.groupby("Код клиента"):
        if cid not in gt_dict: continue
        ts = set(gt_dict[cid])
        if not ts: continue
        recs.append(len(set(group["Поставщик"].values) & ts) / len(ts))
    return np.mean(recs) if recs else 0.0

def ndcg_at_k_graded(predictions, ground_truth, k=3):
    topk = _prepare_topk(predictions, k)
    gt_rel = ground_truth.set_index("Код клиента")["relevance_dict"].to_dict()
    ndcgs = []
    for cid, group in topk.groupby("Код клиента"):
        if cid not in gt_rel: continue
        rel_dict = gt_rel[cid]
        rel = np.array([rel_dict.get(p, 0.0) for p in group["Поставщик"].values])
        dcg = (rel / np.log2(np.arange(2, len(rel) + 2))).sum()
        ideal_rel = np.sort(list(rel_dict.values()))[::-1][:k]
        idcg = (ideal_rel / np.log2(np.arange(2, len(ideal_rel) + 2))).sum()
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
    return np.mean(ndcgs) if ndcgs else 0.0

def map_at_k(predictions, ground_truth, k=3):
    topk = _prepare_topk(predictions, k)
    gt_dict = ground_truth.set_index("Код клиента")["true_suppliers"].to_dict()
    aps = []
    for cid, group in topk.groupby("Код клиента"):
        if cid not in gt_dict: continue
        true_list = gt_dict[cid]
        hits, sum_precs = 0, 0.0
        for i, p in enumerate(group["Поставщик"].values):
            if p in true_list:
                hits += 1
                sum_precs += hits / (i + 1.0)
        aps.append(sum_precs / min(len(true_list), k) if true_list else 0.0)
    return np.mean(aps) if aps else 0.0

## 4. Aligned target

In [10]:
def build_aligned_target(tx_train, cutoff_date, window_days=30,
                         alpha_vol=0.5, alpha_freq=0.5, min_share=0.05):
    cutoff = pd.to_datetime(cutoff_date)
    window_start = cutoff - pd.Timedelta(days=window_days)
    
    tx_window = tx_train[
        (tx_train["Время транзакции"] >= window_start) &
        (tx_train["Время транзакции"] < cutoff)
    ].copy()
    
    if "Поставщик" not in tx_window.columns:
        tx_window["Поставщик"] = tx_window["Тип карты"].str.extract(r'(Поставщик\d+)')
    
    agg = tx_window.groupby(["Код клиента", "Поставщик"]).agg(
        volume=("Объем", "sum"), txn=("Объем", "count")
    ).reset_index()
    
    tot_vol = agg.groupby("Код клиента")["volume"].transform("sum")
    tot_txn = agg.groupby("Код клиента")["txn"].transform("sum")
    agg["vol_share"] = agg["volume"] / tot_vol
    agg["txn_share"] = agg["txn"] / tot_txn
    agg["target"] = alpha_vol * agg["vol_share"] + alpha_freq * agg["txn_share"]
    agg = agg[agg["target"] >= min_share].copy()
    return agg[["Код клиента", "Поставщик", "target"]]

tt_aligned = build_aligned_target(
    tx_train, cutoff_date=SPLIT_DATE,
    window_days=ROLLING_WINDOW_DAYS,
    alpha_vol=ALPHA_VOL, alpha_freq=ALPHA_FREQ, min_share=0.05
)
print(f"aligned target: {len(tt_aligned)} строк, {tt_aligned['Код клиента'].nunique()} клиентов")

aligned target: 4532 строк, 3723 клиентов


In [ ]:
# проверка мерджа aligned target 
print("Колонки tt:", list(tt.columns))
if "Поставщик" in tt.columns:
    print("Уникальные значения 'Поставщик' в tt:", sorted(tt["Поставщик"].unique()))
else:
    print("⚠ В tt нет колонки 'Поставщик'!")
print("Уникальные значения 'Поставщик' в tt_aligned:", sorted(tt_aligned["Поставщик"].unique()))

tt_clean = tt.copy()
if "target" in tt_clean.columns:
    sentinel = tt_clean["target"] <= -1e8
    if sentinel.sum() > 0:
        print(f"Удалено {sentinel.sum()} сентинелей")
        tt_clean = tt_clean[~sentinel].reset_index(drop=True)

tt_no_target = tt_clean.drop(columns=["target"], errors="ignore")
tt_for_catboost = tt_no_target.merge(tt_aligned, on=["Код клиента", "Поставщик"], how="left")
tt_for_catboost["target"] = tt_for_catboost["target"].fillna(0.0)

print(f"\n=== Проверка мерджа ===")
print(f"tt_for_catboost: {len(tt_for_catboost)} строк")
print(f"target > 0: {(tt_for_catboost['target'] > 0).sum()}")
print(f"уникальных клиентов с target > 0: {tt_for_catboost[tt_for_catboost['target']>0]['Код клиента'].nunique()}")

if (tt_for_catboost["target"] > 0).sum() == 0:
    print("\n⚠ КРИТИЧЕСКАЯ ОШИБКА: мердж не сработал — target весь нулевой")
    print("Содержимое первых строк tt:")
    print(tt_clean.head())
    raise RuntimeError("Мердж aligned target не сработал.")
else:
    print("✓ Мердж сработал, target пересчитан")

Колонки tt: ['Код клиента', 'Поставщик', 'region_coverage_pct', 'fuel_match_cosine', 'fuel_match_overlap', 'actual_monthly_volume', 'avg_market_price_per_liter', 'personalized_price_ratio', 'estimated_monthly_saving', 'has_history', 'hard_filter_pass', 'personalized_savings_per_liter', 'composite_score', 'score_economy', 'score_coverage', 'score_fuel', 'score_stability', 'top_factor', 'score_pct', 'expected_savings_pct']
Уникальные значения 'Поставщик' в tt: ['Поставщик1', 'Поставщик2', 'Поставщик3', 'Поставщик4']
Уникальные значения 'Поставщик' в tt_aligned: ['Поставщик1', 'Поставщик2', 'Поставщик3', 'Поставщик4']

=== Проверка мерджа ===
tt_for_catboost: 16340 строк
target > 0: 4532
уникальных клиентов с target > 0: 3723
✓ Мердж сработал, target пересчитан


## 5. Обучение моделей на потенциальных клиентах

In [ ]:
# Фильтрация train для CatBoost 
tt_potential = tt_for_catboost[
    tt_for_catboost["Код клиента"].isin(all_potential_set)
].reset_index(drop=True)

print(f"CatBoost обучается на {tt_potential['Код клиента'].nunique()} потенциальных клиентах")
print(f"Размер обучающей выборки: {len(tt_potential)} пар")

class EASEFixedRecommender:
    def __init__(self, reg=250.0):
        self.base = EASERecommender(reg=reg)
        self.invert = False
    def fit(self, interaction_features):
        self.base.fit(interaction_features)
        sample = interaction_features["Код клиента"].drop_duplicates().head(50).tolist()
        preds = self.base.predict_scores(sample)
        ease_top = preds.sort_values(["Код клиента", "score"], ascending=[True, False]).groupby("Код клиента").head(1)
        actual_top = interaction_features.sort_values(["Код клиента", "volume"], ascending=[True, False]).groupby("Код клиента").head(1)
        merged = ease_top.merge(actual_top, on="Код клиента", suffixes=("_ease", "_actual"))
        agreement = (merged["Поставщик_ease"] == merged["Поставщик_actual"]).mean()
        if agreement < 0.3:
            print(f"   EASE: agreement {agreement:.2f} → инверсия")
            self.invert = True
        return self
    def predict_scores(self, client_ids):
        preds = self.base.predict_scores(client_ids).copy()
        if self.invert:
            preds["score"] = -preds["score"]
        return preds

print("\nОбучение моделей...")
models = {
    "MostPop": MostPopRecommender().fit(tf["interaction_features"]),
    "ALS (k=12)": ALSRecommender(n_factors=12).fit(tf["interaction_features"]),
    "EASE": EASEFixedRecommender(reg=250.0).fit(tf["interaction_features"]),
}

cf_models_for_hybrid = {
    "MostPop": models["MostPop"],
    "ALS": models["ALS (k=12)"], "EASE": models["EASE"]
}

print("\nCatBoost (старый target, всё население)...")
models["CatBoost (old, all)"] = CatBoostLikeRanker(
    n_estimators=300, learning_rate=0.05, cf_models=cf_models_for_hybrid
).fit(tt_clean, tf["compatibility_features"], tf["client_profile"], tf["supplier_profile"])

print("CatBoost (aligned target, только потенциальные)...")
models["CatBoost (aligned, potential)"] = CatBoostLikeRanker(
    n_estimators=300, learning_rate=0.05, cf_models=cf_models_for_hybrid
).fit(tt_potential, tf["compatibility_features"], tf["client_profile"], tf["supplier_profile"])

print("Content-Based...")
models["Content-Based"] = ContentBasedRecommender().fit(tt_clean)

CatBoost обучается на 688 потенциальных клиентах
Размер обучающей выборки: 2752 пар

Обучение моделей...
   EASE: agreement 0.00 → инверсия

CatBoost (старый target, всё население)...


Got unsafe target value = -1e+09 at object #4 of dataset learn


CatBoost (aligned target, только потенциальные)...


Got unsafe target value = -1e+09 at object #65 of dataset learn


Content-Based...


In [13]:
all_eligible = gt["Код клиента"].tolist()
predictions = {}
for name, model in models.items():
    print(f"Предсказание: {name}...")
    if name == "Content-Based":
        preds = model.predict_scores(all_eligible)
    elif name.startswith("CatBoost"):
        preds = model.predict_scores(
            all_eligible, tf["compatibility_features"],
            tf["client_profile"], tf["supplier_profile"]
        )
    else:
        preds = model.predict_scores(all_eligible)
    predictions[name] = preds
print("Готово")

Предсказание: MostPop...
Предсказание: ALS (k=12)...
Предсказание: EASE...
Предсказание: CatBoost (old, all)...
Предсказание: CatBoost (aligned, potential)...
Предсказание: Content-Based...
Готово


## 6. Оценка по сегментам

In [14]:
def evaluate_full(predictions, gt, segment_name="All"):
    results = []
    seg_clients = set(gt["Код клиента"])
    for name, pred in predictions.items():
        pred_seg = pred[pred["Код клиента"].isin(seg_clients)]
        results.append({
            "Сегмент": segment_name, "N клиентов": len(seg_clients),
            "Модель": name,
            "HR@1": round(hit_rate_at_k(pred_seg, gt, k=1), 3),
            "P@2": round(precision_at_k(pred_seg, gt, k=2), 3),
            "R@2": round(recall_at_k(pred_seg, gt, k=2), 3),
            "NDCG@3": round(ndcg_at_k_graded(pred_seg, gt, k=3), 3),
            "MAP@3": round(map_at_k(pred_seg, gt, k=3), 3)
        })
    return pd.DataFrame(results).sort_values("NDCG@3", ascending=False).reset_index(drop=True)

In [15]:
print("="*80)
print("ГЛАВНАЯ ТАБЛИЦА: ПОТЕНЦИАЛЬНЫЕ КЛИЕНТЫ (cold-start + hot+)")
print("Это основной результат V3 — на ком модели должны работать")
print("="*80)
res_potential = evaluate_full(predictions, gt_potential, "Potential")
display(res_potential)

ГЛАВНАЯ ТАБЛИЦА: ПОТЕНЦИАЛЬНЫЕ КЛИЕНТЫ (cold-start + hot+)
Это основной результат V3 — на ком модели должны работать


,Сегмент,N клиентов,Модель,HR@1,P@2,R@2,NDCG@3,MAP@3
0,Potential,402,EASE,0.933,0.614,0.861,0.926,0.913
1,Potential,402,ALS (k=12),0.886,0.679,0.933,0.923,0.917
2,Potential,402,"CatBoost (aligned, potential)",0.898,0.592,0.837,0.905,0.889
3,Potential,402,"CatBoost (old, all)",0.866,0.626,0.869,0.887,0.879
4,Potential,402,MostPop,0.833,0.598,0.826,0.878,0.861
5,Potential,402,Content-Based,0.806,0.597,0.842,0.854,0.831


In [16]:
print("=== Cold-start (< 10 транзакций в train) ===")
if len(gt_cold) > 0:
    res_cold = evaluate_full(predictions, gt_cold, "Cold-start")
    display(res_cold)
else:
    print("Cold-start клиентов в GT не найдено")
    res_cold = None

print("\n=== Hot+ (новый поставщик / смена топ-1 / Δshare ≥ 25%) ===")
print("Меняющееся поведение — настоящий тест предсказательной силы")
if len(gt_hot_plus) > 0:
    res_hot = evaluate_full(predictions, gt_hot_plus, "Hot+")
    display(res_hot)
else:
    print("Hot+ клиентов не найдено")
    res_hot = None

=== Cold-start (< 10 транзакций в train) ===


,Сегмент,N клиентов,Модель,HR@1,P@2,R@2,NDCG@3,MAP@3
0,Cold-start,166,EASE,0.994,0.521,0.983,0.991,0.993
1,Cold-start,166,ALS (k=12),0.982,0.530,0.992,0.984,0.986
2,Cold-start,166,"CatBoost (aligned, potential)",0.958,0.518,0.977,0.969,0.968
3,Cold-start,166,"CatBoost (old, all)",0.910,0.512,0.968,0.949,0.944
4,Cold-start,166,Content-Based,0.855,0.518,0.977,0.928,0.917
5,Cold-start,166,MostPop,0.813,0.491,0.917,0.910,0.889



=== Hot+ (новый поставщик / смена топ-1 / Δshare ≥ 25%) ===
Меняющееся поведение — настоящий тест предсказательной силы


,Сегмент,N клиентов,Модель,HR@1,P@2,R@2,NDCG@3,MAP@3
0,Hot+,244,EASE,0.889,0.680,0.773,0.879,0.858
1,Hot+,244,ALS (k=12),0.816,0.785,0.890,0.877,0.865
2,Hot+,244,MostPop,0.848,0.678,0.764,0.857,0.844
3,Hot+,244,"CatBoost (aligned, potential)",0.857,0.645,0.738,0.855,0.832
4,Hot+,244,"CatBoost (old, all)",0.832,0.703,0.794,0.837,0.829
5,Hot+,244,Content-Based,0.775,0.654,0.746,0.799,0.772


In [17]:
print("=== Контроль: инертные клиенты ===")
if len(gt_inertial) > 0:
    res_inertial = evaluate_full(predictions, gt_inertial, "Inertial")
    display(res_inertial)
else:
    res_inertial = None

print("\n=== Все клиенты (для сравнения с V2) ===")
res_all = evaluate_full(predictions, gt, "All")
display(res_all)

=== Контроль: инертные клиенты ===


,Сегмент,N клиентов,Модель,HR@1,P@2,R@2,NDCG@3,MAP@3
0,Inertial,3134,ALS (k=12),0.987,0.571,0.994,0.984,0.993
1,Inertial,3134,EASE,0.981,0.546,0.969,0.981,0.980
2,Inertial,3134,"CatBoost (old, all)",0.972,0.547,0.968,0.978,0.969
3,Inertial,3134,"CatBoost (aligned, potential)",0.961,0.533,0.951,0.970,0.960
4,Inertial,3134,MostPop,0.857,0.507,0.897,0.921,0.902
5,Inertial,3134,Content-Based,0.761,0.509,0.906,0.877,0.845



=== Все клиенты (для сравнения с V2) ===


,Сегмент,N клиентов,Модель,HR@1,P@2,R@2,NDCG@3,MAP@3
0,All,3536,ALS (k=12),0.976,0.584,0.987,0.977,0.984
1,All,3536,EASE,0.976,0.554,0.956,0.974,0.972
2,All,3536,"CatBoost (old, all)",0.960,0.556,0.957,0.968,0.959
3,All,3536,"CatBoost (aligned, potential)",0.954,0.539,0.938,0.963,0.952
4,All,3536,MostPop,0.855,0.518,0.889,0.916,0.897
5,All,3536,Content-Based,0.766,0.519,0.899,0.875,0.843


In [18]:
summary_segments = {"All": res_all, "Potential": res_potential}
if res_cold is not None: summary_segments["Cold"] = res_cold
if res_hot is not None: summary_segments["Hot+"] = res_hot
if res_inertial is not None: summary_segments["Inertial"] = res_inertial

summary = None
for seg_name, df in summary_segments.items():
    sub = df[["Модель", "NDCG@3"]].rename(columns={"NDCG@3": f"NDCG_{seg_name}"})
    summary = sub if summary is None else summary.merge(sub, on="Модель")

if "NDCG_All" in summary.columns and "NDCG_Potential" in summary.columns:
    summary["Δ_All_vs_Potential"] = (summary["NDCG_All"] - summary["NDCG_Potential"]).round(3)

summary = summary.sort_values("NDCG_Potential", ascending=False).reset_index(drop=True)

print("="*80)
print("СВОДНАЯ: NDCG@3 по сегментам")
print("NDCG_Potential — главный показатель качества модели")
print("Δ — насколько модель теряется при переходе от инерции к потенциалу")
print("="*80)
display(summary)

СВОДНАЯ: NDCG@3 по сегментам
NDCG_Potential — главный показатель качества модели
Δ — насколько модель теряется при переходе от инерции к потенциалу


,Модель,NDCG_All,NDCG_Potential,NDCG_Cold,NDCG_Hot+,NDCG_Inertial,Δ_All_vs_Potential
0,EASE,0.974,0.926,0.991,0.879,0.981,0.048
1,ALS (k=12),0.977,0.923,0.984,0.877,0.984,0.054
2,"CatBoost (aligned, potential)",0.963,0.905,0.969,0.855,0.970,0.058
3,"CatBoost (old, all)",0.968,0.887,0.949,0.837,0.978,0.081
4,MostPop,0.916,0.878,0.910,0.857,0.921,0.038
5,Content-Based,0.875,0.854,0.928,0.799,0.877,0.021
